In [ ]:
#| default_exp stats

# Statistics
> Mars year definitions and temporal classification utilities.

In [ ]:
#| export
import numpy as np
import pandas as pd

from p4tools.io import mars_years, define_martian_year

## Per-tile quality (uncertainty) API

Three layers that separate **support** (vote counts) from **scatter** (marker disagreement). See the *Per-Tile Quality* tutorial for the rationale.

In [ ]:
#| export
def add_uncertainty_columns(df, *, min_votes: int = 5, circular_ratio_limit: float = 0.8):
    """Add derived per-marking uncertainty columns; returns a copy.

    The catalog kind (fan vs blotch) is inferred from the columns present.

    Parameters
    ----------
    df : pandas.DataFrame
        A fan or blotch catalog frame carrying the per-marking cluster standard
        deviations (``x_std``, ``y_std``, ``angle_std`` and, by kind,
        ``distance_std``/``spread_std`` for fans or ``radius1_std``/``radius2_std``
        for blotches) plus ``n_votes``.
    min_votes : int, default 5
        A std estimated from fewer votes is treated as too noisy to use for
        scatter (drives the ``scatter_ok`` gate).
    circular_ratio_limit : float, default 0.8
        Blotches with ``radius_2 / radius_1`` above this are near-circular, so
        their orientation is physically ill-defined and ``angle_usable`` is False.

    Returns
    -------
    pandas.DataFrame
        Copy of ``df`` with added columns ``pos_std``, ``size_cv``,
        ``angle_usable`` and ``scatter_ok``.
    """
    df = df.copy()
    is_blotch = {"radius_1", "radius_2"}.issubset(df.columns)

    # positional scatter: combine the two axes like a 2-D standard deviation
    df["pos_std"] = np.hypot(df["x_std"], df["y_std"])

    # size dispersion as a mean coefficient of variation (relative, scale-free)
    if is_blotch:
        df["size_cv"] = 0.5 * (
            df["radius1_std"] / df["radius_1"] + df["radius2_std"] / df["radius_2"]
        )
        # radius_1 is the major axis, radius_2 the minor, so the ratio is in (0, 1];
        # near 1 means near-circular and the fitted orientation is meaningless
        df["angle_usable"] = (df["radius_2"] / df["radius_1"]) <= circular_ratio_limit
    else:
        df["size_cv"] = 0.5 * (
            df["distance_std"] / df["distance"] + df["spread_std"] / df["spread"]
        )
        df["angle_usable"] = True  # fan orientation is always meaningful

    # support gate: a std estimated from a handful of votes is itself mostly noise
    df["scatter_ok"] = df["n_votes"] >= min_votes
    return df

In [ ]:
#| export
def tile_quality(kind: str = "both", version: str | None = None, *,
                 min_votes: int = 5, agg: str = "median", ranks: bool = True):
    """One row per ``tile_id`` with separate support and scatter column groups.

    The central design rule (see the *Per-Tile Quality* tutorial) is that
    **support** — how many votes back a tile — and **scatter** — how much the
    citizen markers disagreed — are kept on separate axes and never mixed.
    Scatter is computed only from markings that pass the vote gate
    (``n_votes >= min_votes``), so weak support cannot masquerade as
    disagreement. Vote counts never divide the scatter numbers (no SEM).

    Parameters
    ----------
    kind : {"both", "fan", "blotch"}, default "both"
        Which catalog(s) to include.
    version : str, optional
        Catalog version; defaults to the p4tools session version (v3.1).
    min_votes : int, default 5
        Vote gate for the scatter group (passed to `add_uncertainty_columns`).
    agg : {"median", "mean"}, default "median"
        Tile-level aggregator for the scatter metrics. "median" is robust.
    ranks : bool, default True
        Append cap-wide percentile-rank columns ``support_rank``/``scatter_rank``
        (0-100; 50 = median tile). ``scatter_rank`` is the median of the three
        scatter metrics' individual ranks so incommensurate units (deg, px, --)
        combine; ``support_rank`` is the same construction on the vote axis.

    Returns
    -------
    pandas.DataFrame
        Indexed by ``tile_id`` with support columns (``n_markings``, ``n_fans``,
        ``n_blotches``, ``votes_median``, ``votes_min``, ``votes_total``), scatter
        columns (``angle_scatter``, ``pos_scatter``, ``size_scatter``,
        ``n_scatter_markings``) and, when ``ranks``, ``support_rank`` /
        ``scatter_rank``. Scatter values are NaN for tiles with no gated marking.
    """
    from p4tools import io

    if kind not in ("both", "fan", "blotch"):
        raise ValueError(f"kind must be 'both', 'fan' or 'blotch'; got {kind!r}")
    if agg not in ("median", "mean"):
        raise ValueError(f"agg must be 'median' or 'mean'; got {agg!r}")

    frames = []
    if kind in ("fan", "both"):
        f = add_uncertainty_columns(io.get_fan_catalog(version), min_votes=min_votes)
        f["_kind"] = "fan"
        frames.append(f)
    if kind in ("blotch", "both"):
        b = add_uncertainty_columns(io.get_blotch_catalog(version), min_votes=min_votes)
        b["_kind"] = "blotch"
        frames.append(b)
    df = pd.concat(frames, ignore_index=True, sort=False)

    g = df.groupby("tile_id")
    out = pd.DataFrame(index=g.size().index)
    out.index.name = "tile_id"

    # --- support group (pure vote axis; nothing here becomes a scatter number) ---
    out["n_markings"] = g.size()
    out["n_fans"] = (
        df[df._kind == "fan"].groupby("tile_id").size().reindex(out.index, fill_value=0)
    )
    out["n_blotches"] = (
        df[df._kind == "blotch"].groupby("tile_id").size().reindex(out.index, fill_value=0)
    )
    out["votes_median"] = g["n_votes"].median()
    out["votes_min"] = g["n_votes"].min()
    out["votes_total"] = g["n_votes"].sum()

    # --- scatter group (only markings passing the vote gate) ---
    gated = df[df.scatter_ok]
    gg = gated.groupby("tile_id")
    angle_src = gated[gated.angle_usable].groupby("tile_id")["angle_std"].agg(agg)
    out["angle_scatter"] = angle_src.reindex(out.index)
    out["pos_scatter"] = gg["pos_std"].agg(agg).reindex(out.index)
    out["size_scatter"] = gg["size_cv"].agg(agg).reindex(out.index)
    out["n_scatter_markings"] = gg.size().reindex(out.index, fill_value=0).astype(int)

    # --- ranks group (cap-wide percentile position, 0-100) ---
    if ranks:
        def prank(s):
            return s.rank(pct=True) * 100.0
        out["scatter_rank"] = pd.concat(
            [prank(out["angle_scatter"]), prank(out["pos_scatter"]),
             prank(out["size_scatter"])], axis=1
        ).median(axis=1)
        out["support_rank"] = pd.concat(
            [prank(out["n_markings"]), prank(out["votes_median"]),
             prank(out["votes_total"])], axis=1
        ).median(axis=1)
    return out

In [ ]:
#| export
def classify_tile_quality(tq, *, support_thresh: float = 50.0, scatter_thresh: float = 50.0):
    """Add a ``quality_class`` label from the two rank axes.

    Requires ``support_rank`` and ``scatter_rank`` (i.e. ``tile_quality(..., ranks=True)``).
    A tile is "high support" when ``support_rank >= support_thresh`` and
    "high scatter" when ``scatter_rank >= scatter_thresh``.

    ==============  =======  =======  ==========================================
    quality_class   support  scatter  reading
    ==============  =======  =======  ==========================================
    consistent      high     low      reliable -- best QC tier
    contested       high     high     many votes, genuine disagreement
    sparse          low      low      few votes, but they agree
    noisy           low      high     least reliable
    ==============  =======  =======  ==========================================

    Tiles whose ``scatter_rank`` (or ``support_rank``) is NaN -- e.g. no marking
    passed the vote gate, so no scatter estimate exists -- get
    ``quality_class = <NA>`` rather than a guess.

    Parameters
    ----------
    tq : pandas.DataFrame
        Output of `tile_quality` with rank columns.
    support_thresh, scatter_thresh : float, default 50.0
        Percentile-rank thresholds splitting high/low on each axis.

    Returns
    -------
    pandas.DataFrame
        Copy of ``tq`` with an added ``quality_class`` column.
    """
    if "support_rank" not in tq.columns or "scatter_rank" not in tq.columns:
        raise ValueError("tq must come from tile_quality(..., ranks=True)")
    tq = tq.copy()
    hi_support = tq["support_rank"] >= support_thresh
    hi_scatter = tq["scatter_rank"] >= scatter_thresh
    label = pd.Series(pd.NA, index=tq.index, dtype="object")
    label[hi_support & ~hi_scatter] = "consistent"
    label[hi_support & hi_scatter] = "contested"
    label[~hi_support & ~hi_scatter] = "sparse"
    label[~hi_support & hi_scatter] = "noisy"
    # unclassifiable where either rank is missing
    label[tq["support_rank"].isna() | tq["scatter_rank"].isna()] = pd.NA
    tq["quality_class"] = label
    return tq